## Desarrollo de funciones para la Api


+ def **UserForGenre( *`genero` : str* )**:
    Debe devolver el usuario que acumula más horas jugadas para el género dado y una lista de la acumulación de horas jugadas por año.

    Ejemplo de retorno: {"Usuario con más horas jugadas para Género X" : us213ndjss09sdf,
			     "Horas jugadas":[{Año: 2013, Horas: 203}, {Año: 2012, Horas: 100}, {Año: 2011, Horas: 23}]}






In [46]:
#Importar librerias
import pandas as pd
import numpy as np 
import pyarrow as pa
import pyarrow.parquet as pq

### Función **def UserForGenre(genero:str)**:

Debe devolver el usuario que acumula más horas jugadas para el género dado y una lista de la acumulación de horas jugadas por año.

    Ejemplo de retorno: {"Usuario con más horas jugadas para Género X" : us213ndjss09sdf,
			     "Horas jugadas":[{Año: 2013, Horas: 203}, {Año: 2012, Horas: 100}, {Año: 2011, Horas: 23}]}

In [47]:
steam_games= pd.read_csv('../data/steam_7_api.csv', encoding='utf8')
steam_games.head(2)
genre= steam_games[["item_id","genres","Año de lanzamiento"]]
user_items= pd.read_csv('../data/items_3.csv', encoding='utf8')
user_items.head(2)

,item_id,item_name,playtime_forever,user_id,steam_id,items_count
0,10,Counter-Strike,6,76561197970982479,76561197970982479,277
1,20,Team Fortress Classic,0,76561197970982479,76561197970982479,277


In [48]:
user_items2=user_items.copy() #creo una tabla extra por las dudas

In [49]:
user_items2['user_id']=user_items2['user_id'].astype('string') #paso de object a string

In [50]:
play_time_user= user_items[["item_id","playtime_forever","user_id"]] # me quedo con las variables necesarias
play_time_user.head(3)

,item_id,playtime_forever,user_id
0,10,6,76561197970982479
1,20,0,76561197970982479
2,30,7,76561197970982479


In [51]:
genre.head(3) #verifico antes mi tabla a curzar

,item_id,genres,Año de lanzamiento
0,761140,Action,2018
1,761140,Casual,2018
2,761140,Indie,2018


In [52]:
#unifico
genre_user= play_time_user.merge(genre, on="item_id")
genre_user.head(5)

,item_id,playtime_forever,user_id,genres,Año de lanzamiento
0,10,6,76561197970982479,Action,2000
1,10,0,js41637,Action,2000
2,10,0,Riot-Punch,Action,2000
3,10,93,doctr,Action,2000
4,10,108,corrupted_soul,Action,2000


In [53]:
genre_user['genres']=genre_user['genres'].astype('string') #pasar de object a string
genre_user['Año de lanzamiento']=genre_user['Año de lanzamiento'].astype('string')
genre_user['user_id']=genre_user['user_id'].astype('string')

In [54]:
#agrupo el genero por usuario y horas jugadas
user_hours_year= genre_user.groupby(["genres","user_id","Año de lanzamiento"])["playtime_forever"].sum().reset_index()
user_hours_year

,genres,user_id,Año de lanzamiento,playtime_forever
0,Action,--000--,2009,5329
1,Action,--000--,2010,22
2,Action,--000--,2011,6522
3,Action,--000--,2012,109346
4,Action,--000--,2013,363
...,...,...,...,...
3449993,Web Publishing,zepavil,2015,9010
3449994,Web Publishing,zeshirky,2007,1
3449995,Web Publishing,zevlupine,2012,4
3449996,Web Publishing,zilaman,2013,9


In [55]:
#necesito saber los usuarios con mayor horas jugadas por genero
user_hours_total= genre_user.groupby(["genres","user_id"])["playtime_forever"].sum().reset_index()
user_hours_total.head(3)

,genres,user_id,playtime_forever
0,Action,--000--,139469
1,Action,--ace--,69325
2,Action,--ionex--,38315


In [56]:
#por cada genero el total de horas sumadas
max_user = user_hours_total.groupby(["genres"])["playtime_forever"].agg(playtime_forever="max").reset_index()
max_user

,genres,playtime_forever
0,Action,1699307
1,Adventure,2191551
2,Animation &amp; Modeling,168314
3,Audio Production,109916
4,Casual,1224933
5,Design &amp; Illustration,168314
6,Early Access,316969
7,Education,65427
8,Free to Play,808241
9,Indie,2402994


In [57]:
tabla_user =pd.merge(max_user,user_hours_total)
tabla_user #ya tengo el usuario con mayor horas jugadas por genero, ahora me falta incluir el total por año

,genres,playtime_forever,user_id
0,Action,1699307,Sp3ctre
1,Adventure,2191551,REBAS_AS_F-T
2,Animation &amp; Modeling,168314,ScottyG555
3,Audio Production,109916,Lickidactyl
4,Casual,1224933,REBAS_AS_F-T
5,Design &amp; Illustration,168314,ScottyG555
6,Early Access,316969,76561197978756659
7,Education,65427,76561198059330972
8,Free to Play,808241,idonothack
9,Indie,2402994,REBAS_AS_F-T


In [58]:
user_hours_year2=user_hours_year.rename(columns={'playtime_forever':'playtime_forever_year'})
user_hours_year2.head(5)

,genres,user_id,Año de lanzamiento,playtime_forever_year
0,Action,--000--,2009,5329
1,Action,--000--,2010,22
2,Action,--000--,2011,6522
3,Action,--000--,2012,109346
4,Action,--000--,2013,363


In [59]:
#voy a probar si el cruce funciona ok, en el caso del usuario REBAS_AS_F-T	
prueba3=user_hours_year2[(user_hours_year2['user_id'] == 'REBAS_AS_F-T') & (user_hours_year2['genres']=='Indie')]
prueba3
#Para el usuario REBAS_AS_F-T la tabla tabla_user me debe traer 16 registros

,genres,user_id,Año de lanzamiento,playtime_forever_year
1970474,Indie,REBAS_AS_F-T,1999,0
1970475,Indie,REBAS_AS_F-T,2001,11
1970476,Indie,REBAS_AS_F-T,2003,1863
1970477,Indie,REBAS_AS_F-T,2005,0
1970478,Indie,REBAS_AS_F-T,2006,1673
1970479,Indie,REBAS_AS_F-T,2007,1070
1970480,Indie,REBAS_AS_F-T,2008,1366
1970481,Indie,REBAS_AS_F-T,2009,28993
1970482,Indie,REBAS_AS_F-T,2010,21487
1970483,Indie,REBAS_AS_F-T,2011,100155


In [60]:
tabla_user2 =pd.merge(tabla_user,user_hours_year2)
tabla_user2

,genres,playtime_forever,user_id,Año de lanzamiento,playtime_forever_year
0,Action,1699307,Sp3ctre,0,1657
1,Action,1699307,Sp3ctre,1993,0
2,Action,1699307,Sp3ctre,1995,217
3,Action,1699307,Sp3ctre,1996,0
4,Action,1699307,Sp3ctre,1998,0
...,...,...,...,...,...
165,Utilities,207651,76561198073642113,2014,207651
166,Video Production,168314,ScottyG555,2015,168314
167,Web Publishing,142964,Xyphien,2005,7296
168,Web Publishing,142964,Xyphien,2012,64657


In [61]:
#verificamos
prueba4=tabla_user2[(tabla_user2['user_id'] == 'REBAS_AS_F-T') & (tabla_user2['genres']=='Indie')]
prueba4.shape # esta ok
#ya tenemos la tabla para crear la funcion

(16, 5)

In [62]:
tabla_user2 = tabla_user2.rename(columns={'Año de lanzamiento':'Año', 'playtime_forever_year':'Horas jugadas'})
tabla_user2["Año"] = pd.to_numeric(tabla_user2["Año"])

In [63]:
def UserForGenre(genero):
    usuario= tabla_user2[tabla_user2["genres"]== genero]["user_id"].iloc[0] #obtengo usuario
    historial=tabla_user2[(tabla_user2['user_id'] == usuario) & (tabla_user2['genres']==genero)] #filtro por el genero y usuario
    historial2 = historial[['Año', 'Horas jugadas']].copy() #me quedo con las columnas necesarias
    historial3=historial2.to_dict(orient="records")
    return {"Usuario":usuario ,"con más horas jugadas para": genero, "Historial acumulado": historial3 }

In [64]:
UserForGenre('Indie')

{'Usuario': 'REBAS_AS_F-T',
 'con más horas jugadas para': 'Indie',
 'Historial acumulado': [{'Año': 1999, 'Horas jugadas': 0},
  {'Año': 2001, 'Horas jugadas': 11},
  {'Año': 2003, 'Horas jugadas': 1863},
  {'Año': 2005, 'Horas jugadas': 0},
  {'Año': 2006, 'Horas jugadas': 1673},
  {'Año': 2007, 'Horas jugadas': 1070},
  {'Año': 2008, 'Horas jugadas': 1366},
  {'Año': 2009, 'Horas jugadas': 28993},
  {'Año': 2010, 'Horas jugadas': 21487},
  {'Año': 2011, 'Horas jugadas': 100155},
  {'Año': 2012, 'Horas jugadas': 148459},
  {'Año': 2013, 'Horas jugadas': 169349},
  {'Año': 2014, 'Horas jugadas': 326927},
  {'Año': 2015, 'Horas jugadas': 751765},
  {'Año': 2016, 'Horas jugadas': 815989},
  {'Año': 2017, 'Horas jugadas': 33887}]}

In [65]:
#se guarda para la api
tabla_user3 = "funcion.csv"
tabla_user2.to_csv(tabla_user3, index=False, encoding="utf-8")

In [66]:
#guardo parquet
funcion= pd.read_csv("funcion.csv") 

#Indico donde quiero guardar el parquet y con que nombre
output_file= "../data/funcion.parquet"

#Transformo a traves de una tabla el archivo csv en parquet
table = pa.Table.from_pandas(funcion)
pq.write_table(table,output_file)